In [0]:
df_br_brands = spark.read.table("ecommerce.bronze.brz_brands")
df_br_brands.show(5)

#### Data Cleaning + EDA 

In [0]:
# Remove unwanted spaces TRIM
from pyspark.sql.functions import trim , col, regexp_replace
df_br_brands = df_br_brands.withColumn("brand_name", trim(col("brand_name")) )

# Remove special chars from brand_code
df_br_brands = df_br_brands.withColumn("brand_code",regexp_replace(col("brand_code"),r'[^A-Za-z0-9]', ''))

# Select disitinct Brand names 
df_br_brands.select(col("category_code")).distinct().show()

### Replace dummy anaomalies

In [0]:
dd = {
    'BKS' : "BOOKS",
    'GRCY': "GROCERY",
    'TOY' : "TOYS"
}

def get_val(val: str) -> str:
    if val is None:
        return "NA"
    return dd.get(val.strip(), val)

# Register this as UDF :)
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType 

udf_mapped_get_val = udf(get_val, StringType())

df_br_brands = df_br_brands.withColumn("category_code", udf_mapped_get_val(col("category_code")))

In [0]:
df_br_brands.select(col("category_code")).distinct().show()

- Now that it time to write Dataframe as Table we have 2 options 
- A. External table.\
- -  we can do this using save() method (saveAsTable() always created managed tables) \
- - so save() method will always store data in parquet or delta format , so in order to read this table we have to use spark.read.format() plus we will not have actual table in LHS menu. \
- - so how to do when I want table in LHS and also it's files to be stored in cloud ? 
- --------we have to iuse spark.catalog.createTable() \


- B. Managed Table.  
use saveAsTable() and managed table is created. 


In [0]:
spark.sql("DROP TABLE IF EXISTS ecommerce.silver.brz_brands")

In [0]:
# managed Table 
df_br_brands.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.silver.slv_brands")

# External table with table 

# step 1 
## Write dataframe as selta /parq to s3 
# Step 2 
## register as table using spark.catalog.createTable / create table <> using location s3 

name = "slv_brands"
df_br_brands.write.format("delta").mode("overwrite").save("s3://sj-dbr-demo-proj/silver_data/slv_brands")

## IMP - now in above step we created external table now in order to update it we again have to read it and write back 
# so if we want more convinient way where we can see above location pointed table in LHS (Catalog) and want to update data which will intern update delta files then below method :) 

"""
path = "s3://sj-dbr-demo-proj/silver_data/slv_brands"
spark.catalog.createTable(
    "ecommerce.silver.slv_brands_ext", # Catalog path
    path=path,  # points to s3 delta files 
    source="delta",
    description="External table for brands"
)
"""


### you see one drawback of spark.catalog.createTable ? 
- it can not recreate (drop tand recreate table if exists ), we manually have to drop table first before running code block of "spark.catalog.createTable"
- so to overcome this we can use "create or replace table ecommerce.silver.slv_brands_ext
USING DELTA
LOCATION 's3://dummy-dbr-data/slv_brands'
- it has all same features of prev approach + flexibility of dropping :)

In [0]:
spark.sql('drop table if exists ecommerce.silver.slv_brands_ext')

In [0]:
dbutils.notebook.exit("SUCCESS")